# GW170817 PE — TaylorF2 — Fixed sky location (NGC 4993), 128 s segments

Parameter estimation of GW170817 with **sky location fixed** to the known EM counterpart (NGC 4993):
$$\alpha = 3.44616\;\mathrm{rad}, \quad \delta = -0.408084\;\mathrm{rad}$$

- **Waveform**: `TaylorF2` — Post-Newtonian 3.5PN point-particle waveform (no tidal terms), implemented in JAX
- **Sampler**: SHARPy SMC (Sequential Monte Carlo)
- **Data**: 1024 s of GWOSC strain for H1, L1, V1 — L1 glitch subtracted via **BayesWave** ([DCC LIGO-T1700406-v3](https://dcc.ligo.org/LIGO-T1700406/public))
- **Segment duration**: 128 s (matching the paper, $\Delta f \approx 0.0078$ Hz)
- **Frequency range**: $[23, 2000]$ Hz

**Google Colab**: Open this notebook directly from GitHub via `File → Open notebook → GitHub` and paste the repository URL. For **private repos**, use a GitHub personal access token in the Colab GitHub dialog.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saulo-albuquerque-phys/mlgw_bns_jax/blob/jax_mlgw_bns/sharpy_taylorf2_pe_fixedsky.ipynb)

**9 sampled parameters** (RA and Dec fixed; no tidal parameters in TaylorF2 point-particle):

| Index | Parameter | Prior range | Boundary |
|:---:|---|---|---|
| 0 | $\ln d_L$ | $[\ln 1, \ln 75]$ | reflective |
| 1 | $\theta_{JN}$ (inclination) | $[0, \pi]$ | reflective |
| 2 | $\phi_c$ (phase) | $[0, 2\pi]$ | periodic |
| 3 | $\psi$ (polarisation) | $[0, \pi]$ | periodic |
| 4 | $\mathcal{M}_c$ (chirp mass) | $[1.18, 1.21]\,M_\odot$ | reflective |
| 5 | $q$ (mass ratio) | $[0.5, 1.0]$ | reflective |
| 6 | $t_c$ (coalescence time) | $[-0.1, 0.1]\,\mathrm{s}$ | reflective |
| 7 | $\chi_1$ (spin 1) | $[-0.5, 0.5]$ | reflective |
| 8 | $\chi_2$ (spin 2) | $[-0.5, 0.5]$ | reflective |

### Phase convention

The TaylorF2 template uses the standard GW/SPA convention $\tilde{h}(f) \propto e^{+i\Psi(f)}$,
consistent with SHARPy's `project_waveform` which applies $e^{-2\pi i f \cdot \mathrm{timeshift}}$
via the Fourier shift theorem. This matches the IMRPhenomD convention used by ripple/SHARPy.

The spins $\chi_{1,2}$ appear in the prior bounds but are **not used** in the TaylorF2 3.5PN
point-particle phase expansion (no spin-orbit or spin-spin terms). They are sampled to maintain
the same parameter-space structure as the tidal runs.

## Environment setup (Colab / fresh environment)

This cell installs all required packages and clones the repositories. **Skip if running locally** with everything already installed.

In [ ]:
import os, subprocess, sys

COLAB = "google.colab" in sys.modules
REPO_DIR = "/content/mlgw_bns_jax" if COLAB else os.getcwd()

if COLAB:
    # ── Install JAX with CUDA 12 support ─────────────────────────────
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "jax[cuda12]",
        "-f", "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html",
    ])
    # ── Install other Python packages ────────────────────────────────
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "corner", "gwpy", "h5py", "astropy", "netket",
    ])
    # ── Install BlackJAX fork (SHARPy needs custom build_kernel) ─────
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--force-reinstall", "--no-deps",
        "blackjax @ git+https://github.com/gabrieledemasi/blackjax@main",
    ])

    # ── Clone the main repo ──────────────────────────────────────────
    if not os.path.isdir(REPO_DIR):
        subprocess.check_call([
            "git", "clone", "--branch", "jax_mlgw_bns", "--depth", "1",
            "https://github.com/saulo-albuquerque-phys/mlgw_bns_jax.git",
            REPO_DIR,
        ])

    # ── Clone SHARPy ─────────────────────────────────────────────────
    sharpy_repo = os.path.join(REPO_DIR, "_sharpy_repo")
    sharpy_pkg  = os.path.join(sharpy_repo, "sharpy")
    sharpy_link = os.path.join(REPO_DIR, "sharpy")
    if not os.path.isdir(sharpy_repo):
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/saulo-albuquerque-phys/sharpy.git",
            sharpy_repo,
        ])
    if not os.path.exists(sharpy_link):
        os.symlink(sharpy_pkg, sharpy_link)

    # ── Fix blackjax circular import (Python 3.11 compatibility) ─────
    import site
    for _sp in site.getsitepackages():
        _chees = os.path.join(_sp, "blackjax", "adaptation", "chees_adaptation.py")
        if os.path.exists(_chees):
            with open(_chees, "r") as f:
                _csrc = f.read()
            _old_ci = "import blackjax.optimizers.dual_averaging as dual_averaging"
            _new_ci = "from blackjax.optimizers import dual_averaging"
            if _old_ci in _csrc:
                _csrc = _csrc.replace(_old_ci, _new_ci)
                with open(_chees, "w") as f:
                    f.write(_csrc)
                print("Patched chees_adaptation.py: fixed circular import.")
            break

    # ── Fix ripplegw import compatibility in SHARPy ──────────────────
    _gw_lik = os.path.join(sharpy_pkg, "GW_likelihood.py")
    _old_import = "from ripplegw import ms_to_Mc_eta"
    with open(_gw_lik, "r") as f:
        _src = f.read()
    if _old_import in _src and "try:" not in _src.split(_old_import)[0][-30:]:
        _new_import = (
            "try:\n"
            "    from ripplegw import ms_to_Mc_eta\n"
            "except ImportError:\n"
            "    def ms_to_Mc_eta(m):\n"
            "        m1, m2 = m\n"
            "        return (m1 * m2) ** (3 / 5) / (m1 + m2) ** (1 / 5), m1 * m2 / (m1 + m2) ** 2"
        )
        _src = _src.replace(_old_import, _new_import)
        with open(_gw_lik, "w") as f:
            f.write(_src)
        print("Patched GW_likelihood.py: ms_to_Mc_eta import made robust.")

    os.chdir(REPO_DIR)
    print(f"Working directory: {os.getcwd()}")
    print(f"SHARPy: {sharpy_link} -> {sharpy_pkg}")
else:
    print("Not running on Colab — skipping setup.")

## Download GWOSC data (BayesWave-cleaned L1)

Downloads 1024 s of 4 kHz strain from GWOSC for H1, L1 and V1.

For **L1**, the scatter-light glitch near the merger is removed using the
official **BayesWave glitch subtraction** from
[DCC LIGO-T1700406-v3](https://dcc.ligo.org/LIGO-T1700406/public) —
the same cleaned data used for the GWTC-1 parameter estimation
(Abbott+ 2019, PRX 9, 011001).

The cleaned GWF covers GPS ≥ 1187008667 (553 s into our 1024 s window).
We splice: raw L1 for the earlier portion + BayesWave-cleaned for
the rest.

**Skip if the cleaned files already exist.**

In [ ]:
import os, sys, time, shutil
import numpy as np

_GPS_START = 1187008114
_DURATION  = 1024
_SRATE     = 4096
_DATA_DIR  = "gw170817_data"
os.makedirs(_DATA_DIR, exist_ok=True)

_DETECTORS = ["H1", "L1", "V1"]

# ── BayesWave-subtracted L1 data from DCC LIGO-T1700406-v3 ──────────
_DCC_GWF_URL = (
    "https://dcc.ligo.org/public/0144/T1700406/003/"
    "L-L1_CLEANED_HOFT_C02_T1700406_v3-1187008667-4096.gwf"
)
_DCC_CHANNEL = "L1:DCH-CLEAN_STRAIN_C02_T1700406_v3"
_DCC_GPS0    = 1187008667
_DCC_SRATE   = 16384


def _ensure_gwf_backend():
    """Make sure at least one GWF reader is importable."""
    for mod in ("frameCPP", "lalframe", "framel"):
        try:
            __import__(mod)
            return
        except ImportError:
            pass
    conda = shutil.which("conda") or shutil.which("mamba")
    if conda:
        import subprocess
        for pkg in ("framel", "python-lalframe"):
            print(f"  Trying: {conda} install -c conda-forge {pkg}", flush=True)
            ret = subprocess.call([conda, "install", "-c", "conda-forge", "-y", "-q", pkg])
            if ret == 0:
                return
    import subprocess
    for pkg in ("framel",):
        ret = subprocess.call([sys.executable, "-m", "pip", "install", "-q", pkg])
        if ret == 0:
            return
    raise ImportError(
        "Cannot read GWF files. Install a backend manually:\n"
        "  conda install -c conda-forge framel\n"
    )


def _read_gwf_channel(path, channel, start, end):
    """Read a single channel from a GWF file (try gwpy then framel)."""
    try:
        from gwpy.timeseries import TimeSeries
        ts = TimeSeries.read(path, channel, start=start, end=end)
        return np.asarray(ts.value, dtype=np.float64), float(ts.sample_rate.value)
    except Exception:
        pass
    import framel
    vec = framel.frgetvect1d(path, channel, start, end - start, 0)
    data = np.asarray(vec[0], dtype=np.float64)
    sr   = 1.0 / vec[3]
    return data, sr


_all_exist = all(
    os.path.isfile(os.path.join(_DATA_DIR,
        f"{d[0]}-{d}_BWCLEANED_4KHZ-{_GPS_START}-{_DURATION}.txt"))
    for d in _DETECTORS
)

if _all_exist:
    print("Cleaned data files already exist — skipping download.")
else:
    from gwpy.timeseries import TimeSeries
    from scipy.signal import decimate as _decimate

    for det in _DETECTORS:
        out_file = os.path.join(_DATA_DIR,
            f"{det[0]}-{det}_BWCLEANED_4KHZ-{_GPS_START}-{_DURATION}.txt")

        if det == "L1":
            print("L1: building cleaned timeseries")
            t0 = time.time()

            print("  Downloading raw L1 from GWOSC...", flush=True)
            ts_raw = TimeSeries.fetch_open_data(
                "L1", _GPS_START, _GPS_START + _DURATION, sample_rate=_SRATE)

            gwf_local = os.path.join(_DATA_DIR, "L1_cleaned_bw_T1700406.gwf")
            if not os.path.isfile(gwf_local):
                import requests
                print("  Downloading BayesWave GWF from DCC (~1 GB)…", flush=True)
                resp = requests.get(_DCC_GWF_URL, stream=True)
                resp.raise_for_status()
                with open(gwf_local, "wb") as fout:
                    for chunk in resp.iter_content(chunk_size=1 << 20):
                        fout.write(chunk)
                print(f"  Saved GWF ({os.path.getsize(gwf_local)/1e6:.0f} MB)")
            else:
                print("  BayesWave GWF already cached.")

            _ensure_gwf_backend()

            _need_end = _GPS_START + _DURATION
            print("  Reading cleaned segment from GWF...", flush=True)
            bw_data, bw_sr = _read_gwf_channel(
                gwf_local, _DCC_CHANNEL, _DCC_GPS0, _need_end)

            if int(round(bw_sr)) != _SRATE:
                factor = int(round(bw_sr)) // _SRATE
                print(f"  Resampling {int(bw_sr)} -> {_SRATE} Hz (factor {factor})")
                bw_data = _decimate(bw_data, factor, ftype="iir", zero_phase=True)

            n_raw = int((_DCC_GPS0 - _GPS_START) * _SRATE)
            strain = np.concatenate([ts_raw.value[:n_raw], bw_data])
            n_expected = _DURATION * _SRATE
            assert len(strain) == n_expected, (
                f"L1 splice length mismatch: {len(strain)} vs {n_expected}")

            with open(out_file, "w") as fw:
                fw.write("# BayesWave-cleaned L1 strain for GW170817\n")
                fw.write(f"# GPS [{_GPS_START}, {_GPS_START+_DURATION}], "
                         f"splice at GPS {_DCC_GPS0}\n")
                fw.write("# Before splice: raw GWOSC.  After: DCC T1700406-v3 (BayesWave)\n")
                fw.write(f"# {_SRATE} samples per second\n")
                for val in strain:
                    fw.write(f"{val:.16e}\n")
            print(f"  -> L1 done in {time.time()-t0:.1f}s")

        else:
            print(f"{det}: downloading {_DURATION}s from GWOSC…", flush=True)
            t0 = time.time()
            ts = TimeSeries.fetch_open_data(
                det, _GPS_START, _GPS_START + _DURATION, sample_rate=_SRATE)
            with open(out_file, "w") as fw:
                fw.write(f"# {det}: raw GWOSC strain for GW170817\n")
                fw.write(f"# {_SRATE} samples per second\n")
                fw.write(f"# starting GPS {_GPS_START} duration {_DURATION}\n")
                for val in ts.value:
                    fw.write(f"{val:.16e}\n")
            print(f"  -> saved in {time.time()-t0:.1f}s")

    print("All detectors ready.")

In [ ]:
from __future__ import annotations

import os, sys, time
from functools import partial

import numpy as np

# Use GPU if available on Colab, otherwise CPU
if "google.colab" not in sys.modules:
    os.environ.setdefault("JAX_PLATFORMS", "cpu")

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

print("JAX devices:", jax.devices())

## TaylorF2 waveform template and monkey-patch SHARPy

Pure JAX implementation of the **3.5PN point-particle** TaylorF2 frequency-domain waveform,
used as a drop-in replacement for SHARPy's `template()` function.

### Phase convention (important)

The numpy FFT convention uses $e^{-2\pi i f t}$, so the Fourier-domain
waveform uses **negative** phase:
$$\tilde{h}_0(f) = \mathcal{A}(f)\, e^{-i\Psi(f)}$$

This matches ripple/IMRPhenomD (`h0 = A * exp(-i * Psi)`) and SHARPy's
`project_waveform`, which applies the detector time shift as:
$$h_{\text{det}}(f) = \bigl(F_+ h_+ + F_\times h_\times\bigr)\, e^{-2\pi i f \cdot \text{timeshift}}$$

The polarisations follow the ripple convention:
$$h_+ = e^{-2i\phi_c}\, h_0\, \frac{1+\cos^2\iota}{2}, \qquad h_\times = -i\, e^{-2i\phi_c}\, h_0\, \cos\iota$$

**NOTE**: Tidal parameters ($\Lambda_1, \Lambda_2$) and spins ($\chi_1, \chi_2$) are **ignored**
in the phase. This is a pure point-particle, non-spinning approximant at 3.5PN.

In [ ]:
import sharpy.GW_likelihood as _gw_mod
from sharpy.utils import McQ2Masses

from astropy import constants as const
M_sun = const.M_sun.value   # kg
G     = const.G.value        # m^3 kg^-1 s^-2
c     = const.c.value        # m s^-1
pc    = const.pc.value       # m


def TaylorF2_template(params, frequency_array):
    """TaylorF2 3.5PN point-particle waveform (JAX).

    Uses the SHARPy 13-parameter convention:
        [0] ra, [1] dec, [2] logdist, [3] incl, [4] phic, [5] pol,
        [6] mc, [7] q, [8] tc, [9] chi1, [10] chi2, [11] lambda_1, [12] lambda_2

    Phase convention: exp(-i*Psi) — matching ripple/IMRPhenomD and numpy
    FFT convention exp(-2*pi*i*f*t).

    NOTE: Tidal parameters (lambda_1, lambda_2) and spins (chi1, chi2)
    are IGNORED in this point-particle, non-spinning approximant.
    """
    Mc       = params[6]                # chirp mass (M_sun)
    q        = params[7]                # mass ratio m2/m1 <= 1
    phi_c    = params[4]                # coalescence phase
    logdist  = params[2]                # log(distance / Mpc)
    cos_iota = jnp.cos(params[3])       # cos(inclination angle)

    distance = jnp.exp(logdist)         # Mpc
    nu = q / ((1.0 + q) ** 2)           # symmetric mass ratio

    Mc_kg = Mc * M_sun                  # chirp mass in kg
    r = distance * pc * 1e6             # distance in metres
    M = Mc_kg / (nu ** (3.0 / 5.0))    # total mass in kg

    # PN velocity parameter: v = (pi M f)^(1/3) / c
    pi_M = G * jnp.pi * M
    v = jnp.power(pi_M * frequency_array, 1.0 / 3.0) / c
    gamma_e = jnp.float64(0.5772156649015329)  # Euler-Mascheroni constant

    # Newtonian amplitude
    amp = (jnp.power(jnp.pi, -2.0 / 3.0) * jnp.sqrt(5.0 / 24.0)
           * jnp.power(G * Mc_kg / c**3, 5.0 / 6.0)
           * jnp.power(frequency_array, -7.0 / 6.0)
           * (c / r))

    # 3.5PN phase expansion (point-particle, non-spinning)
    v2 = v**2;  v3 = v**3;  v4 = v**4
    v5 = v**5;  v6 = v**6;  v7 = v**7
    log_v = jnp.log(v)

    psi = (3.0 / (128.0 * nu * v5)) * (
        1.0
        + v2 * (20.0 / 9.0) * (743.0 / 336.0 + nu * 11.0 / 4.0)
        - v3 * (16.0 * jnp.pi)
        + v4 * 10.0 * (3058673.0 / 1016064.0
                        + nu * 5429.0 / 1008.0
                        + nu**2 * 617.0 / 144.0)
        + v5 * jnp.pi * (38645.0 / 756.0 - nu * 65.0 / 9.0) * (1.0 + 3.0 * log_v)
        + v6 * (11583231236531.0 / 4694215680.0
                - jnp.pi**2 * 640.0 / 3.0
                - 6848.0 * gamma_e / 21.0
                - 6848.0 / 21.0 * log_v
                + nu * (-15737765635.0 / 3048192.0 + 2255.0 * jnp.pi**2 / 12.0)
                + nu**2 * 76055.0 / 1728.0
                - nu**3 * 127825.0 / 1296.0)
        + v7 * jnp.pi * (77096675.0 / 254016.0
                          + nu * 378515.0 / 1512.0
                          - nu**2 * 74045.0 / 756.0)
    )

    # SPA constant phase offset: -pi/4 from stationary phase integration
    psi -= jnp.pi / 4.0

    cos_iota_sq = cos_iota**2

    # Phase convention: exp(-i*Psi) matching ripple/IMRPhenomD
    # h0 = A * exp(-i*psi), with coalescence phase exp(-2i*phi_c)
    h0 = amp * jnp.exp(-1j * psi)
    phase_factor = jnp.exp(-2j * phi_c)

    h_plus  = phase_factor * h0 * ((1.0 + cos_iota_sq) / 2.0)
    h_cross = phase_factor * (-1j) * h0 * cos_iota

    return h_plus, h_cross


# ── Monkey-patch SHARPy's template ────────────────────────────────────
_gw_mod.template = TaylorF2_template

from sharpy.GW_likelihood import GWNetwork, log_likelihood_det
from sharpy.smc_functions import run_sharpy
import sharpy.PSDs

print("SHARPy template patched with TaylorF2 (3.5PN point-particle, exp(-i*psi) convention).")

## Event parameters

In [ ]:
TRIGGER_TIME = 1187008882.43
SEGMENT_DURATION = 128.0        # paper-matching segment length
SAMPLING_RATE = 4096
F_LOWER = 23.0
F_UPPER = 2000.0
DATA_START_GPS = 1187008114     # 1024s data file start
DATA_DURATION = 1024            # total data length (s)

# Fixed sky location: NGC 4993 (EM counterpart of GW170817)
FIXED_RA  = 3.44616     # rad
FIXED_DEC = -0.408084   # rad

DATA_DIR = "gw170817_data"
OUTDIR = "outdir_GW170817_taylorf2_fixedsky"
LABEL = "GW170817_taylorf2_fixedsky"
os.makedirs(OUTDIR, exist_ok=True)

print(f"Segment duration: {SEGMENT_DURATION}s  →  Δf = {1/SEGMENT_DURATION:.4f} Hz")
print(f"Data: {DATA_DURATION}s starting GPS {DATA_START_GPS}")
print(f"Fixed sky: RA = {FIXED_RA:.5f} rad, Dec = {FIXED_DEC:.6f} rad  (NGC 4993)")

## Load cleaned data and build detector network

We use 1024 s of GWOSC strain data. For L1, the scatter-light glitch near the merger is removed using the official **BayesWave glitch subtraction** from [DCC LIGO-T1700406-v3](https://dcc.ligo.org/LIGO-T1700406/public). H1 and V1 use raw GWOSC data (no significant glitches).

With `SEGMENT_DURATION = 128 s`, SHARPy analyses a 128-s chunk centred on the trigger and uses the remaining ~896 s for Welch PSD estimation (~7 independent segments).

In [ ]:
# BayesWave-cleaned 1024s files (L1 BW-cleaned, H1/V1 raw GWOSC)
data_files = {
    "H1": os.path.join(DATA_DIR, f"H-H1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "L1": os.path.join(DATA_DIR, f"L-L1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "V1": os.path.join(DATA_DIR, f"V-V1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
}
for det, f in data_files.items():
    assert os.path.isfile(f), f"Missing: {f}"
    print(f"{det}: {os.path.basename(f)}")

detector_settings = {}
for det in ["H1", "L1", "V1"]:
    detector_settings[det] = dict(
        data_file=data_files[det], channel="GWOSC",
        trigger_time=TRIGGER_TIME, duration=SEGMENT_DURATION,
        sampling_rate=SAMPLING_RATE,
        f_lower=F_LOWER, f_upper=F_UPPER,
        psd_file=None, psd_method="welch",
        download_data=False, zero_noise=False,
    )

print(f"\nBuilding GW network (segment={SEGMENT_DURATION}s)...")
t0 = time.time()
gw_network = GWNetwork(detector_settings, injection_parameters=None)
print(f"Network built in {time.time() - t0:.2f} s")

## Q-transform spectrograms

Verify the data quality: compare raw vs BayesWave-cleaned L1, and show all three cleaned detectors around the merger time.

In [ ]:
from gwpy.timeseries import TimeSeries
import matplotlib.pyplot as plt

MERGER_GPS = TRIGGER_TIME
WINDOW = 6.0
T_START_PLOT = MERGER_GPS - WINDOW / 2
T_END_PLOT   = MERGER_GPS + WINDOW / 2
F_MIN, F_MAX = 20.0, 800.0
Q_RANGE = (4, 64)

DET_COLORS = {"H1": "Reds", "L1": "Blues", "V1": "Purples"}
DET_LABELS = {"H1": "LIGO Hanford (H1)", "L1": "LIGO Livingston (L1)", "V1": "Virgo (V1)"}

def _qtransform(filepath):
    strain = np.loadtxt(filepath, comments="#")
    ts = TimeSeries(strain, sample_rate=SAMPLING_RATE, t0=DATA_START_GPS)
    ts_w = ts.whiten(4, 2)
    ts_c = ts_w.crop(T_START_PLOT - 1, T_END_PLOT + 1)
    return ts_c.q_transform(frange=(F_MIN, F_MAX), qrange=Q_RANGE,
                            outseg=(T_START_PLOT, T_END_PLOT), logf=True)

# ── L1 raw vs cleaned comparison ─────────────────────────────────────
raw_file = os.path.join(DATA_DIR,
    f"L-L1_GWOSC_4KHZ_R1-{DATA_START_GPS}-{DATA_DURATION}.txt")

if os.path.isfile(raw_file):
    qt_raw = _qtransform(raw_file)
    qt_cln = _qtransform(data_files["L1"])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6), sharey=True)
    for ax, qt, title in [(ax1, qt_raw, "L1 — Raw (with glitch)"),
                           (ax2, qt_cln, "L1 — BayesWave cleaned")]:
        pcm = ax.pcolormesh(qt.times.value - MERGER_GPS, qt.frequencies.value,
                            qt.value.T, cmap="Blues", vmin=0, vmax=25)
        ax.set_yscale("log"); ax.set_ylim(F_MIN, F_MAX)
        ax.set_xlabel("Time relative to merger [s]", fontsize=13)
        ax.set_title(title, fontsize=14)
        ax.axvline(0, color="white", ls="--", lw=1, alpha=0.7, label="Merger")
        ax.legend(loc="upper left"); ax.tick_params(labelsize=11)
        fig.colorbar(pcm, ax=ax).set_label("Normalized energy")
    ax1.set_ylabel("Frequency [Hz]", fontsize=13)
    fig.suptitle("GW170817 — L1 glitch comparison (1024 s data)", fontsize=15)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTDIR, f"{LABEL}_L1_comparison.png"), dpi=150)
    plt.show()
else:
    print(f"Raw L1 file not found ({raw_file}) — skipping glitch comparison.")

# ── All detectors cleaned ────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 14), sharex=True)
for ax, det in zip(axes, ["H1", "L1", "V1"]):
    qt = _qtransform(data_files[det])
    pcm = ax.pcolormesh(qt.times.value - MERGER_GPS, qt.frequencies.value,
                        qt.value.T, cmap=DET_COLORS[det], vmin=0, vmax=25)
    ax.set_yscale("log"); ax.set_ylim(F_MIN, F_MAX)
    ax.set_ylabel("Frequency [Hz]", fontsize=13)
    ax.set_title(f"{DET_LABELS[det]} (BayesWave cleaned)", fontsize=13)
    ax.tick_params(labelsize=11)
    ax.axvline(0, color="white", ls="--", lw=1, alpha=0.7)
    fig.colorbar(pcm, ax=ax).set_label("Normalized energy")
axes[-1].set_xlabel("Time relative to merger [s]", fontsize=13)
fig.suptitle("GW170817 — Q-transform spectrograms (BayesWave-cleaned 1024 s data)",
             fontsize=15, y=0.995)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, f"{LABEL}_qtransform_all.png"), dpi=150)
plt.show()

print("Spectrograms saved.")

## Define likelihood and priors

9 parameters sampled (RA and Dec fixed to NGC 4993), with priors matching the paper:
- Mass prior **flat in component masses** $m_{1,2}$, sampled in $(\mathcal{M}_c, q)$
- Aligned spins $|\chi_{1,2}| \leq 0.5$ (sampled but **not used** in TaylorF2 phase)
- Luminosity distance $D_L \in [1, 75]$ Mpc (flat in $\ln D_L$)
- **No tidal parameters** — TaylorF2 is a point-particle approximant

In [ ]:
batched_detector = gw_network.batched_detector
log_likelihood_full = partial(log_likelihood_det, detector_list=batched_detector)


def log_likelihood_reduced(params_9):
    """Insert fixed RA/Dec and zero tidal params, evaluate full 13-param likelihood.

    params_9 layout:
        [0] logdist, [1] incl, [2] phic, [3] pol,
        [4] mc, [5] q, [6] tc, [7] chi1, [8] chi2
    """
    params_13 = jnp.concatenate([
        jnp.array([FIXED_RA, FIXED_DEC]),     # [0] ra, [1] dec  (fixed)
        params_9[:4],                          # [2] logdist, [3] incl, [4] phic, [5] pol
        params_9[4:9],                         # [6] mc, [7] q, [8] tc, [9] chi1, [10] chi2
        jnp.array([0.0, 0.0]),                 # [11] lambda_1, [12] lambda_2  (unused)
    ])
    return log_likelihood_full(params_13)


# Prior bounds for the 9 sampled parameters
prior_bounds = jnp.array([
    [jnp.log(1.0),  jnp.log(75.0)],     # [0]  logdistance (1–75 Mpc)
    [0.0,           jnp.pi],             # [1]  inclination
    [0.0,           2 * jnp.pi],         # [2]  phic
    [0.0,           jnp.pi],             # [3]  pol
    [1.18,          1.21],               # [4]  mc  (chirp mass, M_sun)
    [0.5,           1.0],                # [5]  q   (mass ratio)
    [-0.1,          0.1],                # [6]  tc  (relative to trigger, s)
    [-0.5,          0.5],                # [7]  chi1
    [-0.5,          0.5],                # [8]  chi2
])

# 1 = periodic, 0 = reflective
boundary_conditions = jnp.array([
    0,  # logdist  (reflective)
    0,  # incl     (reflective)
    1,  # phic     (periodic)
    1,  # pol      (periodic)
    0,  # mc       (reflective)
    0,  # q        (reflective)
    0,  # tc       (reflective)
    0,  # chi1     (reflective)
    0,  # chi2     (reflective)
])

parameter_names = [
    "logdistance", "theta_jn", "phiref", "pol",
    "mc", "q", "tc", "chi1", "chi2",
]


def prior(params):
    """Uniform prior (log-prior = 0 inside bounds)."""
    return 0.0


print(f"Fixed: RA = {FIXED_RA:.5f}, Dec = {FIXED_DEC:.6f}")
print(f"Sampling {len(parameter_names)} parameters: {parameter_names}")

In [ ]:
# Quick sanity check: likelihood at literature values
# GW170817 literature: mc~1.186, q~0.87, D~40 Mpc, iota~2.5 rad
# params_9: [logdist, incl, phic, pol, mc, q, tc, chi1, chi2]

# JIT-compile to avoid repeated tracing overhead
log_likelihood_jit = jax.jit(log_likelihood_reduced)

# Warm-up (single JIT compilation)
_warmup = log_likelihood_jit(jnp.array([
    jnp.log(40.0), jnp.pi/2, 0.0, 0.0,
    1.186, 0.87, 0.0, 0.0, 0.0
]))
_warmup.block_until_ready()
print("JIT warm-up done.")

# Evaluate noise-only logL (reference: params far from signal)
logL_noise = float(log_likelihood_jit(jnp.array([
    jnp.log(75.0), jnp.pi/2, 0.0, 0.0,
    1.21, 1.0, 0.1, 0.0, 0.0
])))

# Reduced grid search (5*4*3*3*3 = 540 evaluations)
import gc
best_logL = -jnp.inf
best_params = None
for mc in [1.186, 1.190, 1.195, 1.197, 1.200]:
    for q in [0.80, 0.87, 0.95, 1.0]:
        for logd in [jnp.log(30.), jnp.log(40.), jnp.log(50.)]:
            for incl in [jnp.pi/2, 2.5, 2.9]:
                for tc in [-0.005, 0.0, 0.005]:
                    p = jnp.array([logd, incl, 0.0, 0.0, mc, q, tc, 0.0, 0.0])
                    ll = float(log_likelihood_jit(p))
                    if ll > best_logL:
                        best_logL = ll
                        best_params = p
    gc.collect()

print(f"Noise logL (reference): {logL_noise:.2f}")
print(f"Best logL found: {best_logL:.2f}")
print(f"Delta logL: {best_logL - logL_noise:.2f}")
print(f"Best params: mc={float(best_params[4]):.4f}, q={float(best_params[5]):.2f}, "
      f"D={jnp.exp(float(best_params[0])):.1f} Mpc, incl={float(best_params[1]):.3f} rad, "
      f"tc={float(best_params[6]):.4f}")

## Run the SMC sampler

With 9 parameters (no tidal deformability), TaylorF2 PE is relatively lightweight compared
to the full BNS run.

In [ ]:
N_PARTICLES = 500
STEP_SIZE = 0.3
ALPHA = 0.95
SEED = 42

print(f"Starting SHARPy SMC with {N_PARTICLES} particles over {len(parameter_names)} parameters...")
start = time.time()

result_dict = run_sharpy(
    log_likelihood_reduced, prior,
    prior_bounds, boundary_conditions,
    ALPHA, N_PARTICLES, STEP_SIZE,
    jax.random.PRNGKey(SEED),
    folder=OUTDIR, label=LABEL,
)

dt = time.time() - start
samples = result_dict["posterior_samples"]
logZ, dlogZ = result_dict["logZ"], result_dict["dlogZ"]
print(f"\nDone in {dt:.1f} s — log Z = {logZ:.2f} ± {dlogZ:.2f}")

## Corner plot

In [ ]:
from corner import corner

fig = corner(
    np.array(samples), show_titles=True,
    labels=parameter_names, title_kwargs={"fontsize": 10},
)
plot_path = os.path.join(OUTDIR, f"{LABEL}_corner.png")
fig.savefig(plot_path, dpi=150)
print(f"Saved to {plot_path}")
fig

## Paper-style corner plot

Compute derived parameters from the posterior samples and produce a corner plot showing:
- $\mathcal{M}_c$ — chirp mass
- $q$ — mass ratio
- $\chi_\text{eff}$ — effective spin parameter
- $D_L$ — luminosity distance [Mpc]

No $\tilde{\Lambda}$ since TaylorF2 does not include tidal effects.

Column indices for the 9-parameter samples:
`[0] logdist, [1] incl, [2] phic, [3] pol, [4] mc, [5] q, [6] tc, [7] chi1, [8] chi2`

In [ ]:
from sharpy.utils import McQ2Masses

# Extract raw sampled parameters (9-param layout)
mc_samples   = np.array(samples[:, 4])   # chirp mass
q_samples    = np.array(samples[:, 5])   # mass ratio (m2/m1 <= 1)
chi1_samples = np.array(samples[:, 7])   # spin 1
chi2_samples = np.array(samples[:, 8])   # spin 2
logd_samples = np.array(samples[:, 0])   # log distance

# Compute component masses
m1_samples = np.zeros(len(mc_samples))
m2_samples = np.zeros(len(mc_samples))
for i in range(len(mc_samples)):
    m1_samples[i], m2_samples[i] = McQ2Masses(mc_samples[i], q_samples[i])

# chi_eff = (m1*chi1 + m2*chi2) / (m1 + m2)
chi_eff_samples = (m1_samples * chi1_samples + m2_samples * chi2_samples) / (m1_samples + m2_samples)

# D_L in Mpc
dL_samples = np.exp(logd_samples)

# Build the 4-parameter array for the corner plot
paper_samples = np.column_stack([
    mc_samples,
    q_samples,
    chi_eff_samples,
    dL_samples,
])

paper_labels = [
    r"$\mathcal{M}_c$ $[M_\odot]$",
    r"$q$",
    r"$\chi_{\rm eff}$",
    r"$D_L$ [Mpc]",
]

fig_paper = corner(
    paper_samples, show_titles=True,
    labels=paper_labels,
    title_kwargs={"fontsize": 12},
    quantiles=[0.05, 0.5, 0.95],
    levels=(0.5, 0.9),
    fill_contours=True,
    color="tab:blue",
)
plot_path_paper = os.path.join(OUTDIR, f"{LABEL}_corner_paper.png")
fig_paper.savefig(plot_path_paper, dpi=150)
print(f"Saved to {plot_path_paper}")
fig_paper